In [ ]:
from pathlib import Path
EXPECTED_GIT_COMMIT = "REPLACE_AFTER_PUSH"
assert (len(EXPECTED_GIT_COMMIT) == 40 and EXPECTED_GIT_COMMIT != "REPLACE_AFTER_PUSH"), "Pin the reviewed pushed commit before Colab validation"
REPO_URL = "https://github.com/sfczaa/ddpm-derm-augmentation.git"
RUN_VERSION = "v1_panderm_base_c1_finetune"
# Operational metadata only: whichever Google account runs this notebook sets
# its own label. It is never part of the immutable experiment identity and it
# never splits the single per-run-version validation lock.
ACCOUNT_LABEL = "A"
assert ACCOUNT_LABEL in {"A", "B", "C"}, "ACCOUNT_LABEL must be A, B, or C"
ARCH = "panderm_base_vit_b16"
CHECKPOINT_FORMAT = "panderm_full_model_v1"
VARIANT = "C1"
DF_TARGET_COUNT = 585
VALIDATION_EPOCHS = 5
BATCH_SIZE = 16
ACCUMULATION_STEPS = 8
EFFECTIVE_BATCH_SIZE = 128
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.05
WARMUP_EPOCHS = 10
LAYER_DECAY = 0.65
DROP_PATH = 0.2
PANDERM_UPSTREAM_REPO = "https://github.com/SiyuanYan1/PanDerm"
PANDERM_UPSTREAM_COMMIT = "fd7a80748ba7fc3e203fed88f909f4689d0d6f24"
PANDERM_CHECKPOINT_FILENAME = "panderm_bb_data6_checkpoint-499.pth"
PANDERM_CHECKPOINT_DRIVE_ID = "removed-from-public-history"
SHARED_PROJECT_DIR = Path("/content/drive/MyDrive/ddpm-derm-augmentation")
SHARED_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-panderm-runs")
COCA_RUN_ROOT = Path("/content/drive/MyDrive/ddpm-derm-coca-runs")
V1_ROOT = SHARED_RUN_ROOT / RUN_VERSION
VALIDATION_RUNS_ROOT = V1_ROOT / "validation_runs"
VALIDATION_RECORD = V1_ROOT / "validation_record.json"
LATEST_FAILURE_RECORD = V1_ROOT / "latest_validation_failure.json"
FORMAL_ROOT = V1_ROOT / "formal"
SHARED_ROOT_SENTINEL = SHARED_RUN_ROOT / ".panderm_shared_root.json"
DATA_CACHE_DIRECTORY = SHARED_RUN_ROOT / "data_cache" / "ham10000_train_val_only_v1"
CODE_DIR = Path("/content/panderm-code")
UPSTREAM_DIR = Path("/content/panderm-upstream")
WEIGHTS_CACHE = Path("/content/panderm-weights")
LOCAL_DATA_DIR = Path("/content/ham10000-data")
SMOKE_SAMPLE_DIR = Path("/content/panderm-train-smoke-sample")

# PanDerm-Base C1 full fine-tuning validation

Prerequisites: both shared Drive shortcuts must resolve physically to the reviewed roots, the user must have Editor permission, and the Colab Secret GH_TOKEN must be enabled for this notebook.

- Project shortcut: `/content/drive/MyDrive/ddpm-derm-augmentation`
- Durable run shortcut: `/content/drive/MyDrive/ddpm-derm-panderm-runs`

Never create a private folder of the same name when either shared shortcut is missing.

Validation-only, fail-fast workflow. Formal training, test access, deployment,
and creation of a second validation attempt remain prohibited.

## Phase 0 CHECK - Drive, roots, sentinel, pinned clones, dependencies, checkpoint SHA, isolation

In [ ]:
import base64, hashlib, json, os, shutil, subprocess, sys, time
from google.colab import drive, userdata
drive.mount("/content/drive")
assert SHARED_PROJECT_DIR.is_dir(), f"missing shared project shortcut: {SHARED_PROJECT_DIR}"
assert SHARED_RUN_ROOT.is_dir(), f"missing shared run shortcut; do not create a private replacement: {SHARED_RUN_ROOT}"
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""): digest.update(chunk)
    return digest.hexdigest()
subprocess.run(["nvidia-smi"], check=True)
token = userdata.get("GH_TOKEN")
assert token and len(token) > 20, "Colab Secret GH_TOKEN with read access to this repo is required (enable notebook access for accounts A/B/C)"
GH_TOKEN_PRESENT = True
assert not CODE_DIR.exists(), f"fresh runtime required: {CODE_DIR}"
basic_credential = base64.b64encode(("x-access-token:" + token).encode()).decode()
clone_env = os.environ.copy()
clone_env["GIT_CONFIG_COUNT"] = "1"
clone_env["GIT_CONFIG_KEY_0"] = "http.https://github.com/.extraheader"
clone_env["GIT_CONFIG_VALUE_0"] = "Authorization: Basic " + basic_credential
try:
    subprocess.run(["git", "clone", REPO_URL, str(CODE_DIR)], check=True, env=clone_env)
finally:
    clone_env["GIT_CONFIG_VALUE_0"] = ""
    token = basic_credential = None
    del token, basic_credential, clone_env
subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--detach", EXPECTED_GIT_COMMIT], check=True)
commit = subprocess.check_output(["git", "-C", str(CODE_DIR), "rev-parse", "HEAD"], text=True).strip()
remote = subprocess.check_output(["git", "-C", str(CODE_DIR), "remote", "get-url", "origin"], text=True).strip()
status = subprocess.check_output(["git", "-C", str(CODE_DIR), "status", "--short"], text=True).strip()
assert commit == EXPECTED_GIT_COMMIT and not status, "clone must be a clean detached checkout of the pinned commit"
assert "@" not in remote and "x-access-token" not in remote, "clone URL must not embed a credential"
assert not UPSTREAM_DIR.exists(), f"fresh runtime required: {UPSTREAM_DIR}"
subprocess.run(["git", "clone", "--filter=blob:none", PANDERM_UPSTREAM_REPO, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", "--detach", PANDERM_UPSTREAM_COMMIT], check=True)
upstream_commit = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "rev-parse", "HEAD"], text=True).strip()
upstream_status = subprocess.check_output(["git", "-C", str(UPSTREAM_DIR), "status", "--short"], text=True).strip()
assert upstream_commit == PANDERM_UPSTREAM_COMMIT and not upstream_status, "PanDerm upstream must be a clean detached checkout of the pinned commit"
os.environ["HF_HOME"] = "/content/hf-cache"
os.environ["TORCH_HOME"] = "/content/torch-cache"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "timm==0.9.16", "gdown>=5.1", "pandas>=2.0", "pillow>=9.0"], check=True)
sys.path.insert(0, str(CODE_DIR / "src"))
from importlib.metadata import version
import torch
from ddpm_derm import panderm_run
assert torch.cuda.is_available() and version("timm") == "0.9.16"
assert panderm_run.UPSTREAM_COMMIT == PANDERM_UPSTREAM_COMMIT == upstream_commit
assert panderm_run.RUN_VERSION == RUN_VERSION and panderm_run.ARCH == ARCH
resolved_root = panderm_run.require_existing_shared_root(SHARED_RUN_ROOT)
drive_probe = panderm_run.probe_shared_drive(resolved_root)
assert SHARED_ROOT_SENTINEL.is_file(), f"missing shared-root sentinel; do not recreate it: {SHARED_ROOT_SENTINEL}"
sentinel = panderm_run.create_or_validate_sentinel(SHARED_ROOT_SENTINEL, shortcut_alias="ddpm-derm-panderm-runs", resolved_path=str(resolved_root), drive_folder_id=None, run_version=RUN_VERSION)
panderm_run.require_shared_root_sentinel_identity(sentinel, shortcut_alias="ddpm-derm-panderm-runs", resolved_root=resolved_root, run_version=RUN_VERSION)
assert SHARED_RUN_ROOT != COCA_RUN_ROOT and COCA_RUN_ROOT not in SHARED_RUN_ROOT.parents, "PanDerm output root must be fully isolated from the CoCa runs"
coca_guard_paths = sorted(COCA_RUN_ROOT.rglob("validation_record.json")) + sorted(COCA_RUN_ROOT.rglob("_COMPLETED.json")) if COCA_RUN_ROOT.is_dir() else []
project_guard_paths = [SHARED_PROJECT_DIR / name for name in ("README.md", "HANDOFF.md", "EXPERIMENT_LOG.md")]
existing_attempts = sorted(path for path in VALIDATION_RUNS_ROOT.iterdir() if path.is_dir()) if VALIDATION_RUNS_ROOT.is_dir() else []
assert len(existing_attempts) <= 1, f"only one validation attempt directory is permitted: {existing_attempts}"
attempt_guard_paths = sorted(path for attempt in existing_attempts for path in attempt.rglob("*") if path.is_file())
unexpected_formal_names = {"last.pt", "best.pt", "last.pt.integrity.json", "best.pt.integrity.json", "_COMPLETED.json", "validation_record.json"}
unexpected_attempt_artifacts = [path for path in attempt_guard_paths if path.name in unexpected_formal_names or (path.parent.name == ARCH and path.name.startswith("results_") and path.suffix == ".json")]
assert not unexpected_attempt_artifacts, f"unexpected resumable/formal artifacts in the unique attempt; inspect manually: {unexpected_attempt_artifacts}"
guard_paths = [path for path in coca_guard_paths + project_guard_paths + attempt_guard_paths if path.is_file()]
before_guard = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
panderm_run.require_no_deployment_contamination(CODE_DIR)
assert not V1_ROOT.is_symlink(), f"validation version parent must not be a symlink: {V1_ROOT}"
if V1_ROOT.exists():
    assert V1_ROOT.is_dir(), f"validation version parent must be a directory: {V1_ROOT}"
    preexisting_v1_root = V1_ROOT.resolve(strict=True)
    assert preexisting_v1_root.is_relative_to(resolved_root), f"validation version parent escaped the shared root: {preexisting_v1_root}"
    assert preexisting_v1_root.parent.samefile(resolved_root), f"validation version parent must be a direct child of the shared root: {preexisting_v1_root}"
    assert not VALIDATION_RECORD.exists(), f"a validation_record already exists; do not re-draw validation for the same version: {VALIDATION_RECORD}"
    assert not LATEST_FAILURE_RECORD.exists(), f"a prior gate already failed this version: {LATEST_FAILURE_RECORD}"
    assert not FORMAL_ROOT.exists(), f"formal artifacts must not exist before validation: {FORMAL_ROOT}"
print(json.dumps({"commit": commit, "upstream_commit": upstream_commit, "drive_probe": drive_probe, "guard_files": len(guard_paths), "existing_attempts": [path.name for path in existing_attempts]}, indent=2))

### Phase 0 CHECK - official checkpoint download and pinned SHA-256

In [ ]:
WEIGHTS_CACHE.mkdir(parents=True, exist_ok=True)
CHECKPOINT_PATH = WEIGHTS_CACHE / PANDERM_CHECKPOINT_FILENAME
if not CHECKPOINT_PATH.is_file():
    import gdown
    gdown.download(id=PANDERM_CHECKPOINT_DRIVE_ID, output=str(CHECKPOINT_PATH), quiet=False)
assert CHECKPOINT_PATH.is_file(), f"PanDerm checkpoint download failed: {CHECKPOINT_PATH}"
observed_checkpoint_sha256 = sha256(CHECKPOINT_PATH)
print("checkpoint bytes:", CHECKPOINT_PATH.stat().st_size)
print("checkpoint sha256:", observed_checkpoint_sha256)
print("expected (pinned):", panderm_run.EXPECTED_CHECKPOINT_SHA256)
checkpoint_sha256 = panderm_run.require_checkpoint_sha256(CHECKPOINT_PATH)
provenance = panderm_run.summarize_provenance()
assert provenance["license_review"]["license"] == "CC-BY-NC-ND-4.0"
assert provenance["license_review"]["finetuning_allowed"] is True
assert provenance["license_review"]["deployment_allowed"] is False
assert provenance["license_review"]["sharing_adapted_weights_allowed"] is False
assert provenance["contamination_review"]["image_level_ham10000_overlap"] == "not_independently_excludable"
assert provenance["contamination_review"]["independent_audit_possible"] is False
assert provenance["contamination_review"]["exact_fixed_validation_test_overlap"] == "unproven"
assert provenance["contamination_review"]["patient_level_overlap"] == "not_excludable"
assert provenance["contamination_review"]["ham10000_in_upstream_finetuning_or_evaluation"] == "yes_evaluation_benchmark"
assert provenance["contamination_review"]["loaded_checkpoint_is_pretraining_only"] is True
assert provenance["claim_boundary"] == "suggestive_exploratory_only"
assert provenance["deployment_allowed"] is False
provenance_clearance = panderm_run.require_provenance_clearance(upstream_commit=upstream_commit, checkpoint_sha256=checkpoint_sha256, purpose=panderm_run.VALIDATION_ONLY)
print(json.dumps(provenance_clearance, indent=2))

## Phase 1 CHECK - checkpoint layout, real CPU model load, primitive run identity and dependency versions

In [ ]:
import gc
assert "DDPM_DERM_DATA_DIR" not in os.environ, "Phase 1 must not bind a training dataset"
from ddpm_derm import panderm
phase1_started = time.monotonic()
print("[Phase 1] START checkpoint/model/identity checks", flush=True)
phase1_factory = panderm.load_upstream_model_factory(UPSTREAM_DIR)
pretrained_state = panderm.load_pretrained_state(CHECKPOINT_PATH)
pretrained_layout = panderm.detect_checkpoint_layout(pretrained_state)
assert pretrained_layout == panderm.LAYOUT_DIRECT_BACKBONE
remapped_state = panderm.remap_pretrained_state_dict(pretrained_state, layout=pretrained_layout)
assert "fc_norm.weight" in remapped_state and "fc_norm.bias" in remapped_state
assert not any(key.startswith(("norm.", "head.")) for key in remapped_state)
train_transform = panderm.build_train_transform()
eval_transform = panderm.build_eval_transform()
preflight_model = panderm.build_panderm_classifier(checkpoint_path=CHECKPOINT_PATH, upstream_dir=UPSTREAM_DIR, drop_path=DROP_PATH)
assert preflight_model.pretrained_state_layout == panderm.LAYOUT_DIRECT_BACKBONE
assert preflight_model.head.out_features == 7 and preflight_model.head.in_features == 768
model_details = panderm.model_identity(preflight_model, train_transform=train_transform, eval_transform=eval_transform, checkpoint_sha256=checkpoint_sha256)
dependency_versions = panderm.dependency_versions()
assert type(dependency_versions["torch"]) is str
manifest_sha256 = {split: sha256(SHARED_PROJECT_DIR / "data" / "manifests" / f"{split}.csv") for split in ("train", "val")}
class_mapping_sha256 = sha256(SHARED_PROJECT_DIR / "data" / "manifests" / "class_to_idx.json")
fixed_split_identity = manifest_sha256["train"]
formal_output_identity = f"{sentinel['shared_root_uuid']}:{RUN_VERSION}:formal"
os.environ["DDPM_DERM_DATA_DIR"] = str(SHARED_PROJECT_DIR / "data")
from ddpm_derm import config, manifests, classifier_objective, train_panderm
preflight_args = train_panderm.parse_args([
    "--variant", VARIANT, "--seed", "0", "--epochs", str(VALIDATION_EPOCHS),
    "--batch-size", str(BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS),
    "--lr", str(LEARNING_RATE), "--weight-decay", str(WEIGHT_DECAY),
    "--warmup-epochs", str(VALIDATION_EPOCHS), "--layer-decay", str(LAYER_DECAY),
    "--df-target-count", str(DF_TARGET_COUNT), "--checkpoint", str(CHECKPOINT_PATH),
    "--checkpoint-sha256", checkpoint_sha256, "--upstream-dir", str(UPSTREAM_DIR),
    "--upstream-commit", upstream_commit, "--evaluation-scope", "validation_only",
    "--output-dir", "/content/panderm-preflight-output", "--run-version", RUN_VERSION,
    "--shared-root-uuid", sentinel["shared_root_uuid"],
    "--formal-output-identity", formal_output_identity,
    "--fixed-split-identity", fixed_split_identity,
])
preflight_optimizer = panderm.build_optimizer(preflight_model, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY)
assert panderm.verify_optimizer_covers_parameters_once(preflight_optimizer, preflight_model) == sum(1 for parameter in preflight_model.parameters() if parameter.requires_grad)
preflight_num_batches = (panderm_run.EXPECTED_C1_TRAIN_ROWS + BATCH_SIZE - 1) // BATCH_SIZE
preflight_steps_per_epoch = panderm.optimizer_steps_per_epoch(preflight_num_batches, ACCUMULATION_STEPS)
assert preflight_num_batches == 469 and preflight_steps_per_epoch == 58
preflight_schedule = panderm.WarmupCosineSchedule(preflight_optimizer, base_lr=LEARNING_RATE, warmup_epochs=VALIDATION_EPOCHS, epochs=VALIDATION_EPOCHS, steps_per_epoch=preflight_steps_per_epoch)
preflight_scaler = torch.amp.GradScaler("cuda", enabled=True)
production_run_identity = panderm_run.build_run_identity(
    git_commit=commit, seed=0, epochs=VALIDATION_EPOCHS,
    evaluation_scope="validation_only", checkpoint_sha256=checkpoint_sha256,
    model_identity=model_details, manifest_sha256=manifest_sha256,
    fixed_split_identity=fixed_split_identity,
    shared_root_uuid=sentinel["shared_root_uuid"],
    formal_output_identity=formal_output_identity,
    dependency_versions=dependency_versions, warmup_epochs=VALIDATION_EPOCHS,
    drop_path=DROP_PATH, amp_requested=True, amp_effective=True,
    device_type="cuda",
)
panderm_run.require_primitive_identity(production_run_identity)
assert json.loads(json.dumps(production_run_identity, sort_keys=True)) == production_run_identity
print(json.dumps({"factory": phase1_factory.__name__, "layout": pretrained_layout, "dependency_versions": dependency_versions, "run_identity_primitive": True}, indent=2))
del pretrained_state, remapped_state
gc.collect()
print(f"[Phase 1] COMPLETE elapsed={time.monotonic() - phase1_started:.1f}s", flush=True)

## Phase 2 CHECK - dataset-free production checkpoint save and weights-only reopen

In [ ]:
assert not LOCAL_DATA_DIR.exists(), "Phase 2 requires a fresh runtime-local data destination"
phase2_started = time.monotonic()
checkpoint_preflight = train_panderm.checkpoint_serialization_preflight(
    temporary_directory=Path("/content"),
    model=preflight_model,
    optimizer=preflight_optimizer,
    schedule=preflight_schedule,
    scaler=preflight_scaler,
    args=preflight_args,
    run_identity=production_run_identity,
)
assert checkpoint_preflight["weights_only_round_trip"] is True
assert checkpoint_preflight["checkpoint_format"] == CHECKPOINT_FORMAT
assert not list(Path("/content").glob(".panderm-checkpoint-preflight.*.pt"))
del preflight_model, preflight_optimizer, preflight_schedule, preflight_scaler
gc.collect()
print(json.dumps(checkpoint_preflight, indent=2))
print(f"[Phase 2] serialization COMPLETE elapsed={time.monotonic() - phase2_started:.1f}s", flush=True)

import queue, threading
test_env = os.environ.copy()
test_env.pop("DDPM_DERM_DATA_DIR", None)
test_env["PYTHONPATH"] = str(CODE_DIR / "src")
test_env["PYTHONUNBUFFERED"] = "1"
test_env["PYTHONDONTWRITEBYTECODE"] = "1"
def run_stream(command, cwd=CODE_DIR, expect_success=True, process_env=None):
    started = time.monotonic()
    process = subprocess.Popen(command, cwd=cwd, env=process_env or test_env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    output_queue = queue.Queue()
    def pump_output():
        for line in process.stdout: output_queue.put(line)
        output_queue.put(None)
    threading.Thread(target=pump_output, daemon=True).start()
    while True:
        try: line = output_queue.get(timeout=60)
        except queue.Empty:
            print(f"[subprocess] heartbeat elapsed={time.monotonic() - started:.1f}s pid={process.pid} alive={process.poll() is None}", flush=True)
            continue
        if line is None: break
        lines.append(line); print(line, end="", flush=True)
    code_returned = process.wait()
    if expect_success and code_returned: raise subprocess.CalledProcessError(code_returned, command)
    if not expect_success and not code_returned: raise AssertionError("command unexpectedly succeeded")
    return time.monotonic() - started, "".join(lines)
_, targeted_output = run_stream([sys.executable, "-B", "-u", "-m", "unittest", "-v", "tests.test_panderm_blockers", "tests.test_panderm_base_c1_finetune", "tests.test_panderm_notebooks", "tests.test_panderm_fresh_runtime"])
_, suite_output = run_stream([sys.executable, "-B", "-u", "-m", "unittest", "discover", "-s", "tests", "-v"])
_, runner_smoke_output = run_stream([sys.executable, "-B", "-u", "-m", "unittest", "-v", "tests.test_panderm_blockers.PanDermRunnerMockSmokeTests"])
targeted_checks = {"targeted_ok": "OK" in targeted_output, "suite_ok": "OK" in suite_output, "runner_smoke_ok": "OK" in runner_smoke_output}
assert all(targeted_checks.values()), targeted_checks
illegal = [
    ["--variant", "C4"], ["--generated-manifest", "/tmp/nope.csv"],
    ["--run-version", "v2_other"], ["--df-target-count", "586"],
    ["--accumulation-steps", "0"], ["--batch-size", "0"],
    ["--warmup-epochs", "999"], ["--evaluation-scope", "full"],
    ["--drop-path", "0.3"], ["--no-amp"], ["--seed", "1"],
    ["--epochs", "50"],
]
base_cli = [sys.executable, "-B", "-u", "-m", "ddpm_derm.train_panderm", "--seed", "0", "--epochs", "5", "--warmup-epochs", "5", "--checkpoint", str(CHECKPOINT_PATH), "--upstream-dir", str(UPSTREAM_DIR), "--output-dir", "/content/panderm-illegal"]
_, base_legal_output = run_stream(base_cli + ["--checkpoint-sha256", "0" * 64], expect_success=False)
assert "SHA-256 mismatch" in base_legal_output
for extra in illegal: run_stream(base_cli + extra, expect_success=False)
print("CLI illegal combinations rejected:", len(illegal))

## Phase 3 CHECK - manifest-only counts, C1 construction and train/val leakage

In [ ]:
import pandas as pd
phase3_started = time.monotonic()
shared_manifests = SHARED_PROJECT_DIR / "data" / "manifests"
frames = {split: manifests.load_split(split) for split in ("train", "val")}
assert len(frames["train"]) == 6995 and len(frames["val"]) == 1510
assert int((frames["train"]["dx"] == "df").sum()) == 85 and int((frames["val"]["dx"] == "df").sum()) == 14
assert not set(frames["train"]["lesion_id"]) & set(frames["val"]["lesion_id"])
assert not set(frames["train"]["image_id"]) & set(frames["val"]["image_id"])
assert manifest_sha256 == {split: sha256(shared_manifests / f"{split}.csv") for split in ("train", "val")}
c1_frame = manifests.build_classifier_frame("C1", df_target_count=DF_TARGET_COUNT, seed=0)
c1_counts = classifier_objective.ordered_class_counts(c1_frame)
assert len(c1_frame) == panderm_run.EXPECTED_C1_TRAIN_ROWS == 7495
assert c1_counts == panderm_run.EXPECTED_C1_CLASS_COUNTS
assert c1_counts["df"] == DF_TARGET_COUNT
assert "source" not in c1_frame.columns or set(c1_frame["source"].unique()) <= {"real"}
real_df_ids = set(frames["train"].loc[frames["train"]["dx"] == "df", "image_id"].astype(str))
c1_df_ids = set(c1_frame.loc[c1_frame["dx"] == "df", "image_id"].astype(str))
assert c1_df_ids == real_df_ids
assert not c1_df_ids & set(frames["val"]["image_id"].astype(str))
assert not LOCAL_DATA_DIR.exists(), "full local staging must remain unreachable in Phase 3"
print(json.dumps({"train_rows": len(frames["train"]), "val_rows": len(frames["val"]), "c1_rows": len(c1_frame), "c1_counts": c1_counts, "manifest_sha256": manifest_sha256}, indent=2))
print(f"[Phase 3] COMPLETE elapsed={time.monotonic() - phase3_started:.1f}s", flush=True)

## Phase 4 CHECK - fixed four-image train sample, real CUDA update and post-step checkpoint round-trip

In [ ]:
from PIL import Image
phase4_started = time.monotonic()
print("[Phase 4] START four-image CUDA update smoke", flush=True)
assert checkpoint_preflight["weights_only_round_trip"] is True
sample_report = panderm_run.stage_train_smoke_sample(SHARED_PROJECT_DIR / "data", SMOKE_SAMPLE_DIR, count=4)
assert sample_report["source_manifest"] == "manifests/train.csv"
assert sample_report["sample_count"] == 4 and sample_report["test_manifest_read"] is False
sample_paths = [SMOKE_SAMPLE_DIR / Path(*relative.split("/")) for relative in sample_report["relative_paths"]]
model = panderm.build_panderm_classifier(checkpoint_path=CHECKPOINT_PATH, upstream_dir=UPSTREAM_DIR, drop_path=DROP_PATH).cuda()
backbone_count = panderm.assert_full_trainability(model)
total_params, trainable_params = panderm.parameter_counts(model)
assert total_params > trainable_params and model.pos_embed.requires_grad is False
train_panderm.set_seed(0)
sample_tensors = [train_transform(Image.open(path).convert("RGB")) for path in sample_paths]
batch = torch.stack(sample_tensors).cuda()
assert tuple(batch.shape) == (4, 3, 224, 224)
model.train()
logits = model(batch)
assert tuple(logits.shape) == (4, 7) and torch.isfinite(logits).all()
optimizer = panderm.build_optimizer(model, learning_rate=LEARNING_RATE, weight_decay=WEIGHT_DECAY, layer_decay=LAYER_DECAY)
covered = panderm.verify_optimizer_covers_parameters_once(optimizer, model)
assert covered == sum(1 for parameter in model.parameters() if parameter.requires_grad)
schedule = panderm.WarmupCosineSchedule(optimizer, base_lr=LEARNING_RATE, warmup_epochs=VALIDATION_EPOCHS, epochs=VALIDATION_EPOCHS, steps_per_epoch=preflight_steps_per_epoch)
scaler = torch.amp.GradScaler("cuda", enabled=True)
before_parameters = panderm.snapshot_parameters(model)
targets = torch.tensor([0, 3, 5, 6], device="cuda")
criterion = torch.nn.CrossEntropyLoss()
model.zero_grad(set_to_none=True)
for micro in range(ACCUMULATION_STEPS):
    with torch.autocast(device_type="cuda", enabled=True):
        loss = criterion(model(batch), targets)
    assert torch.isfinite(loss)
    scaler.scale(loss * panderm.accumulation_loss_scale(micro, ACCUMULATION_STEPS, ACCUMULATION_STEPS)).backward()
    print(f"[Phase 4] accumulation micro-step={micro + 1}/{ACCUMULATION_STEPS} loss={float(loss):.4f}", flush=True)
gradient_report = panderm.backbone_gradient_report(model)
assert gradient_report["blocks_with_gradient"] == list(range(12))
assert gradient_report["all_backbone_blocks_have_gradient"]
assert gradient_report["head_parameters_have_gradient"]
assert gradient_report["fixed_backbone_parameter_names"] == ["pos_embed"]
assert model.pos_embed.grad is None
scaler.unscale_(optimizer)
panderm.require_finite_gradients(model.parameters())
scaler.step(optimizer)
scaler.update()
optimizer.zero_grad(set_to_none=True)
schedule.step()
changed_backbone = panderm.changed_parameter_count(before_parameters, model)
assert changed_backbone > 0
post_step_checkpoint_preflight = train_panderm.checkpoint_serialization_preflight(
    temporary_directory=Path("/content"), model=model, optimizer=optimizer,
    schedule=schedule, scaler=scaler, args=preflight_args,
    run_identity=production_run_identity,
)
assert post_step_checkpoint_preflight["weights_only_round_trip"] is True
backbone_gradient_verified = True
backbone_parameters_updated = True
GPU_SMOKE_COMPLETE = True
del model, optimizer, schedule, scaler, batch, logits, before_parameters, sample_tensors, targets, loss, criterion
gc.collect()
torch.cuda.empty_cache()
shutil.rmtree(SMOKE_SAMPLE_DIR)
gpu_smoke = {
    "official_preprocessing": True,
    "official_checkpoint_loaded": True,
    "cuda_forward": True,
    "backward": True,
    "all_12_blocks_have_gradients": True,
    "head_has_gradients": True,
    "fixed_pos_embed_has_no_gradient": True,
    "optimizer_coverage": True,
    "optimizer_step": True,
    "backbone_updated": True,
    "post_step_checkpoint_round_trip": True,
    "smoke_model_discarded": True,
}
PHASE3_COMPLETE = True
print(json.dumps({"sample": sample_report, "gpu_smoke": gpu_smoke, "changed_backbone_tensors": changed_backbone, "gradient_report": gradient_report}, indent=2))
print(f"[Phase 4] COMPLETE elapsed={time.monotonic() - phase4_started:.1f}s", flush=True)
assert not V1_ROOT.is_symlink(), f"validation version parent must not be a symlink: {V1_ROOT}"
try:
    verified_v1_root = panderm_run.ensure_tree(
        SHARED_RUN_ROOT, V1_ROOT.relative_to(SHARED_RUN_ROOT)
    )
except FileExistsError:
    # A concurrent account may have won the same child mkdir; re-run every
    # safe-tree visibility/write probe instead of weakening the contract.
    verified_v1_root = panderm_run.ensure_tree(
        SHARED_RUN_ROOT, V1_ROOT.relative_to(SHARED_RUN_ROOT)
    )
assert verified_v1_root.is_dir() and not verified_v1_root.is_symlink(), f"validation version parent is not a real directory: {verified_v1_root}"
resolved_v1_root = verified_v1_root.resolve(strict=True)
assert resolved_v1_root.is_relative_to(resolved_root), f"validation version parent escaped the shared root: {resolved_v1_root}"
assert resolved_v1_root.parent.samefile(resolved_root), f"validation version parent must be a direct child of the shared root: {resolved_v1_root}"
VALIDATION_RUN_LOCK = panderm_run.validation_run_lock_path(SHARED_RUN_ROOT)
assert VALIDATION_RUN_LOCK.parent.samefile(verified_v1_root), "validation lock parent does not match V1_ROOT"
assert VALIDATION_RUN_LOCK.parent.resolve(strict=True) == resolved_v1_root, "validation lock parent identity drift"
VALIDATION_SESSION = panderm_run.session_marker(ACCOUNT_LABEL, "fresh", {"run_version": RUN_VERSION})
VALIDATION_SESSION_ID = VALIDATION_SESSION["session_id"]
VALIDATION_LOCK_HELD = False
# Atomic exclusive create. Two accounts that both pass every preflight must not
# both proceed, so no archive, attempt directory or runner call precedes it.
# Run all NEVER clears a stale lock; recovery is the separate manual cell.
print("[run-lock] acquiring after all validation preflights", flush=True)
VALIDATION_RUN_LOCK_MARKER = panderm_run.acquire_validation_run_lock(
    VALIDATION_RUN_LOCK,
    session_id=VALIDATION_SESSION_ID,
    run_version=RUN_VERSION,
    git_commit=commit,
    shared_root_uuid=sentinel["shared_root_uuid"],
    account_label=ACCOUNT_LABEL,
)
assert VALIDATION_RUN_LOCK_MARKER["session_id"] == VALIDATION_SESSION_ID
VALIDATION_LOCK_HELD = True
print("[run-lock] acquired by session", VALIDATION_SESSION_ID, flush=True)

## Phase 5 RUN - verified immutable train/val-only archive staging

In [ ]:
assert checkpoint_preflight["weights_only_round_trip"] is True
assert post_step_checkpoint_preflight["weights_only_round_trip"] is True
assert PHASE3_COMPLETE is True
APPROVED_CONTENT_IDENTITY = panderm_run.require_approved_content_identity(
    panderm_run.EXPECTED_VALIDATION_CONTENT_IDENTITY_SHA256
)
assert APPROVED_CONTENT_IDENTITY != panderm_run.VALIDATION_CONTENT_IDENTITY_PLACEHOLDER
assert len(APPROVED_CONTENT_IDENTITY) == 64
data_cache_parent = panderm_run.ensure_tree(SHARED_RUN_ROOT, Path("data_cache"))
assert DATA_CACHE_DIRECTORY.parent.resolve(strict=True) == data_cache_parent.resolve(strict=True)
assert VALIDATION_LOCK_HELD is True, "archive staging requires the validation run lock"
assert panderm_run._read_validation_run_lock(VALIDATION_RUN_LOCK)["session_id"] == VALIDATION_SESSION_ID
def stage_from_verified_archive():
    if not DATA_CACHE_DIRECTORY.exists():
        panderm_run.build_validation_archive_cache(
            SHARED_PROJECT_DIR / "data", DATA_CACHE_DIRECTORY, Path("/content"),
            expected_file_content_identity_sha256=APPROVED_CONTENT_IDENTITY,
            source_fixed_split_identity=fixed_split_identity,
        )
    return panderm_run.reuse_validation_archive_cache(
        DATA_CACHE_DIRECTORY, Path("/content"), LOCAL_DATA_DIR,
        expected_file_content_identity_sha256=APPROVED_CONTENT_IDENTITY,
        expected_fixed_split_identity=fixed_split_identity,
        expected_manifest_sha256=manifest_sha256,
        expected_class_mapping_sha256=class_mapping_sha256,
    )
staging_report = panderm_run.stage_after_validation_preflights(
    checkpoint_preflight=checkpoint_preflight,
    gpu_smoke=gpu_smoke,
    staging=stage_from_verified_archive,
)
assert staging_report["archive_files_copied"] == 1
assert staging_report["images_staged"] == 8505
assert staging_report["test_manifest_present"] is False
assert staging_report["whole_data_copy_used"] is False
assert staging_report["approved_file_content_identity_sha256"] == APPROVED_CONTENT_IDENTITY
for required_manifest in ("train.csv", "val.csv", "class_to_idx.json"):
    assert (LOCAL_DATA_DIR / "manifests" / required_manifest).is_file()
assert not (LOCAL_DATA_DIR / "manifests" / "test.csv").exists()
archive_identity = panderm_run.validate_validation_archive_cache(
    DATA_CACHE_DIRECTORY,
    expected_file_content_identity_sha256=APPROVED_CONTENT_IDENTITY,
    expected_fixed_split_identity=fixed_split_identity,
    expected_manifest_sha256=manifest_sha256,
    expected_class_mapping_sha256=class_mapping_sha256,
)
training_env = os.environ.copy()
training_env["DDPM_DERM_DATA_DIR"] = str(LOCAL_DATA_DIR)
training_env["PYTHONPATH"] = str(CODE_DIR / "src")
training_env["PYTHONUNBUFFERED"] = "1"
training_env["PYTHONDONTWRITEBYTECODE"] = "1"
assert archive_identity["file_content_identity_sha256"] == APPROVED_CONTENT_IDENTITY
print(json.dumps({"staging_report": staging_report, "archive_identity": archive_identity, "approved_content_identity": APPROVED_CONTENT_IDENTITY}, indent=2))

## Phase 6 RUN - fresh seed-0 five-epoch validation-only subprocess

In [ ]:
import copy
from datetime import datetime, timezone
panderm_run.ensure_tree(SHARED_RUN_ROOT, V1_ROOT.relative_to(SHARED_RUN_ROOT))
panderm_run.ensure_tree(SHARED_RUN_ROOT, VALIDATION_RUNS_ROOT.relative_to(SHARED_RUN_ROOT))
assert VALIDATION_LOCK_HELD is True, "attempt creation requires the validation run lock"
assert panderm_run._read_validation_run_lock(VALIDATION_RUN_LOCK)["session_id"] == VALIDATION_SESSION_ID, "the run lock changed owner; refusing to create an attempt"
existing_attempts = sorted(path for path in VALIDATION_RUNS_ROOT.iterdir() if path.is_dir())
assert len(existing_attempts) <= 1, f"only one validation attempt directory is permitted: {existing_attempts}"
if existing_attempts:
    VALIDATION_DIR = existing_attempts[0]
    validation_id = VALIDATION_DIR.name
else:
    validation_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    VALIDATION_DIR = panderm_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_RUNS_ROOT / validation_id).relative_to(SHARED_RUN_ROOT))
GATE_ROOT = panderm_run.ensure_tree(SHARED_RUN_ROOT, (VALIDATION_DIR / "non_collapse_gate").relative_to(SHARED_RUN_ROOT))
gate_command = [sys.executable, "-B", "-u", "-m", "ddpm_derm.train_panderm", "--variant", VARIANT, "--seed", "0", "--epochs", str(VALIDATION_EPOCHS), "--batch-size", str(BATCH_SIZE), "--accumulation-steps", str(ACCUMULATION_STEPS), "--lr", str(LEARNING_RATE), "--weight-decay", str(WEIGHT_DECAY), "--warmup-epochs", str(VALIDATION_EPOCHS), "--layer-decay", str(LAYER_DECAY), "--df-target-count", str(DF_TARGET_COUNT), "--checkpoint", str(CHECKPOINT_PATH), "--checkpoint-sha256", checkpoint_sha256, "--upstream-dir", str(UPSTREAM_DIR), "--upstream-commit", upstream_commit, "--evaluation-scope", "validation_only", "--output-dir", str(GATE_ROOT), "--run-version", RUN_VERSION, "--shared-root-uuid", sentinel["shared_root_uuid"], "--formal-output-identity", formal_output_identity, "--fixed-split-identity", fixed_split_identity]
checkpoint_dir = GATE_ROOT / "checkpoints" / ARCH / f"{VARIANT}_seed0"
result_path = GATE_ROOT / "results" / ARCH / f"results_{VARIANT}_seed0.json"
unexpected_before_fresh = [checkpoint_dir / "last.pt", checkpoint_dir / "best.pt", checkpoint_dir / "last.pt.integrity.json", checkpoint_dir / "best.pt.integrity.json", result_path]
assert not [path for path in unexpected_before_fresh if path.exists()], "unexpected checkpoint/result exists; do not auto-resume or overwrite"
gate_seconds, gate_output = run_stream(gate_command, process_env=training_env)
assert "[start] fresh run (no --resume) from epoch 1" in gate_output
assert "[test]" not in gate_output and "checkpoint_saved=last.pt" in gate_output
result = json.loads(result_path.read_text(encoding="utf-8"))
panderm_run.require_matching_identity(result["run_identity"], production_run_identity)
assert result["evaluation_scope"] == "validation_only" and result["test_metrics"] is None
assert result["data_counts"] == {"train": 7495, "val": 1510, "test": None}
assert len(result["history"]) == VALIDATION_EPOCHS
best_path, last_path = checkpoint_dir / "best.pt", checkpoint_dir / "last.pt"
assert best_path.is_file() and last_path.is_file()
for path in (best_path, last_path):
    reopened = train_panderm.load_checkpoint_safe(path, map_location="cpu", expected_identity=result["run_identity"], expected_result_checkpoint=result["checkpoint_integrity"][path.name])
    assert reopened["checkpoint_format"] == CHECKPOINT_FORMAT
    assert "model_state_dict" in reopened and "head_state_dict" not in reopened
    assert set(reopened) >= {"optimizer_state_dict", "scheduler_state_dict", "scaler_state_dict", "rng_state", "run_identity"}
    assert reopened["run_identity"] == result["run_identity"]
    panderm_run.require_matching_identity(reopened["run_identity"], result["run_identity"])
assert train_panderm.load_checkpoint_safe(last_path, map_location="cpu", expected_identity=result["run_identity"], expected_result_checkpoint=result["checkpoint_integrity"]["last.pt"])["epoch"] == VALIDATION_EPOCHS
best_checkpoint, last_checkpoint = train_panderm.load_completed_checkpoint_pair_safe(best_path=best_path, last_path=last_path, result=result, model=None, expected_identity=result["run_identity"], map_location="cpu")
panderm_run.require_completed_artifact_identities(expected=result["run_identity"], result=result, best_checkpoint=best_checkpoint, last_checkpoint=last_checkpoint)
before_state = (sha256(last_path), last_path.stat().st_mtime_ns)
_, resumed_output = run_stream(gate_command + ["--resume"], process_env=training_env)
assert "[resume]" in resumed_output and "[skip]" in resumed_output
for index, replacement in (("--layer-decay", "0.75"), ("--lr", "1e-4"), ("--weight-decay", "0.01"), ("--accumulation-steps", "4"), ("--batch-size", "8")):
    mismatch = gate_command.copy()
    mismatch[mismatch.index(index) + 1] = replacement
    run_stream(mismatch + ["--resume"], expect_success=False, process_env=training_env)
saved_identity = last_checkpoint["run_identity"]
for key, value in (("checkpoint_sha256", "0" * 64), ("upstream_commit", "1" * 40), ("claim_boundary", "confirmed"), ("evaluation_scope", "full")):
    drifted = copy.deepcopy(saved_identity)
    drifted[key] = value
    try: panderm_run.require_matching_identity(saved_identity, drifted); raise AssertionError(f"identity drift accepted: {key}")
    except ValueError: pass
truncated = {key: value for key, value in saved_identity.items() if key != "objective"}
try: panderm_run.require_expected_identity_complete(truncated); raise AssertionError("truncated expectation accepted")
except ValueError: pass
assert (sha256(last_path), last_path.stat().st_mtime_ns) == before_state, "a rejected resume must not mutate the checkpoint"
child_code = "import subprocess,sys; subprocess.run(sys.argv[1:], check=True)"
subprocess.run([sys.executable, "-B", "-u", "-c", child_code] + gate_command + ["--resume"], cwd=CODE_DIR, env=training_env, check=True)
resume_checks = {"resume": True, "mismatches_rejected_without_mutation": True, "child_process_drive_only_restore": True}
print(json.dumps({"validation_id": validation_id, "gate_seconds": gate_seconds, "best_val_df_f1": result["best_val_df_f1"], "resume_checks": resume_checks}, indent=2))

## Phase 7 REVIEW - result, best/last reopen, resume, mismatch rejection and non-collapse gate

In [ ]:
predicted = torch.tensor(result["validation_metrics"]["confusion_matrix"]).sum(dim=0)
prediction_counts = {name: int(predicted[index]) for index, name in enumerate(config.CLASS_NAMES)}
identity_complete = set(result["run_identity"]) == set(panderm_run.IMMUTABLE_IDENTITY_KEYS)
trainer_source = (CODE_DIR / "src" / "ddpm_derm" / "train_panderm.py").read_text(encoding="utf-8")
validation_notebook = json.loads((CODE_DIR / "notebooks" / "colab_panderm_base_c1_finetune_validation.ipynb").read_text(encoding="utf-8"))
validation_code = "\n".join("".join(cell.get("source", [])) for cell in validation_notebook["cells"] if cell["cell_type"] == "code")
panderm_run.validate_validation_notebook_source(validation_code)
formal_notebook = json.loads((CODE_DIR / "notebooks" / "colab_panderm_base_c1_finetune_classifier.ipynb").read_text(encoding="utf-8"))
formal_code = "\n".join("".join(cell.get("source", [])) for cell in formal_notebook["cells"] if cell["cell_type"] == "code")
source_guards = {
    "trainer_has_no_test_split_load": 'load_split("test")' not in trainer_source,
    "trainer_has_no_full_scope": 'evaluation_scope == "full"' not in trainer_source,
    "validation_notebook_uses_archive_build": "panderm_run.build_validation_archive_cache" in validation_code,
    "validation_notebook_uses_single_tar_reuse": "panderm_run.reuse_validation_archive_cache" in validation_code,
    "validation_notebook_has_no_legacy_full_copy": "panderm_run.stage_validation_data" not in validation_code,
    "validation_notebook_copy_ast_safe": True,
    "formal_notebook_has_no_test_split_load": 'load_split("test")' not in formal_code,
    "formal_notebook_has_no_training_entrypoint": "ddpm_derm.train_panderm" not in formal_code,
}
test_access_probes = {}
for scenario in ("before_validation_match", "before_three_seed_completion", "forged_validation_pass"):
    try:
        panderm_run.require_provenance_clearance(upstream_commit=upstream_commit, checkpoint_sha256=checkpoint_sha256, purpose=panderm_run.TEST_ACCESS)
    except ValueError as error:
        test_access_probes[scenario] = str(error) == panderm_run.PROHIBITED_FORMAL_TEST_REASON
    else:
        test_access_probes[scenario] = False
no_test_access = all(source_guards.values()) and all(test_access_probes.values()) and result["test_metrics"] is None and result["data_counts"]["test"] is None
checks = panderm_run.evaluate_non_collapse_gate(result=result, prediction_counts=prediction_counts, backbone_gradient_verified=backbone_gradient_verified, backbone_parameters_updated=backbone_parameters_updated, identity_complete=identity_complete, no_test_access=no_test_access, provenance_allows_next_stage=bool(provenance_clearance))
assert set(checks) == set(panderm_run.NON_COLLAPSE_CHECK_KEYS)
gate_metrics = {"best_validation_df_f1": result["best_val_df_f1"], "best_epoch": max(result["history"], key=lambda item: item["val_df_f1"])["epoch"], "history": result["history"], "prediction_counts": prediction_counts, "checks": checks, "test_access_probes": test_access_probes, "source_guards": source_guards, "elapsed_seconds": gate_seconds, "changed_backbone_tensors": changed_backbone, "gradient_report": gradient_report}
gate_failures = [key for key, value in checks.items() if not value]
if gate_failures:
    failure_guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
    assert failure_guard_after == before_guard, "pre-existing read-only guard files changed"
    failure_record = {"validation_status": "VALIDATION FAILED", "formal_training_started": False, "formal_training_allowed": False, "test_access_allowed": False, "run_version": RUN_VERSION, "git_commit": commit, "upstream_commit": upstream_commit, "checkpoint_sha256": checkpoint_sha256, "arch": ARCH, "variant": VARIANT, "gate_failures": gate_failures, "non_collapse_gate": gate_metrics, "model_identity": model_details, "fixed_split_identity": fixed_split_identity, "manifest_sha256": manifest_sha256, "shared_root_uuid": sentinel["shared_root_uuid"], "provenance": provenance, "claim_boundary": panderm_run.CLAIM_BOUNDARY, "evaluation_scope": "validation_only", "test_metrics": None, "validation_artifact_directory": str(VALIDATION_DIR)}
    panderm_run.write_json_atomic(VALIDATION_DIR / "validation_failure.json", failure_record)
    panderm_run.write_json_atomic(LATEST_FAILURE_RECORD, failure_record)
    assert not VALIDATION_RECORD.exists()
    print("VALIDATION FAILED")
    print("formal_training_allowed=false")
    print("test_access_allowed=false")
    panderm_run.release_validation_run_lock(VALIDATION_RUN_LOCK, session_id=VALIDATION_SESSION_ID)
    VALIDATION_LOCK_HELD = False
    print("[run-lock] released after caught failure by", VALIDATION_SESSION_ID, flush=True)
    raise RuntimeError(gate_failures)
print(json.dumps(checks, indent=2))

### Phase 7 REVIEW - immutable validation record and formal/test hard stop

In [ ]:
guard_after = {str(path): (sha256(path), path.stat().st_mtime_ns) for path in guard_paths}
assert guard_after == before_guard, "pre-existing read-only guard files changed"
panderm_run.require_no_deployment_contamination(CODE_DIR)
record = {"validation_status": "VALIDATION PASSED", "formal_training_started": False, "formal_training_allowed": False, "test_access_allowed": False, "run_version": RUN_VERSION, "arch": ARCH, "variant": VARIANT, "git_commit": commit, "upstream_repo": PANDERM_UPSTREAM_REPO, "upstream_commit": upstream_commit, "checkpoint_filename": PANDERM_CHECKPOINT_FILENAME, "checkpoint_source_url": panderm_run.CHECKPOINT_SOURCE_URL, "checkpoint_sha256": checkpoint_sha256, "checkpoint_sha256_provenance": panderm_run.CHECKPOINT_SHA256_PROVENANCE, "checkpoint_bytes": CHECKPOINT_PATH.stat().st_size, "checkpoint_format": CHECKPOINT_FORMAT, "model_identity": model_details, "dependency_versions": dependency_versions, "fixed_split_identity": fixed_split_identity, "manifest_sha256": manifest_sha256, "data_cache_identity": archive_identity, "c1_construction": {"strategy": "duplicate_real_train_df", "df_target_count": DF_TARGET_COUNT, "synthetic_images_used": False, "class_counts": c1_counts, "train_rows": len(c1_frame)}, "objective": result["run_identity"]["objective"], "optimization": result["run_identity"]["optimization"], "shared_root_uuid": sentinel["shared_root_uuid"], "shared_root_identity": sentinel, "formal_output_identity": formal_output_identity, "evaluation_scope": "validation_only", "non_collapse_gate": gate_metrics, "resume_mismatch_checks": resume_checks, "targeted_check_results": targeted_checks, "drive_probes": drive_probe, "provenance": provenance, "license_review": provenance["license_review"], "contamination_review": provenance["contamination_review"], "claim_boundary": panderm_run.CLAIM_BOUNDARY, "results_grade": "exploratory", "deployment_allowed": False, "attribution": panderm_run.ATTRIBUTION, "validation_artifact_directory": str(VALIDATION_DIR), "existing_artifacts_unchanged": True, "gh_token_present": GH_TOKEN_PRESENT, "test_metrics": None}
panderm_run.write_json_atomic(VALIDATION_DIR / "validation_record.json", record)
panderm_run.write_json_atomic(VALIDATION_RECORD, record)
assert not FORMAL_ROOT.exists(), "PanDerm v1 must never create formal artifacts"
assert json.loads(VALIDATION_RECORD.read_text(encoding="utf-8")) == record
# Result, checkpoints and validation record are verified above; only now is
# the lock released, and only because this session owns it.
panderm_run.release_validation_run_lock(VALIDATION_RUN_LOCK, session_id=VALIDATION_SESSION_ID)
VALIDATION_LOCK_HELD = False
print("[run-lock] released by", VALIDATION_SESSION_ID, flush=True)
print(json.dumps(record, indent=2))
print("VALIDATION PASSED")
print("formal_training_allowed=false")
print("test_access_allowed=false")
print("run_version=v1_panderm_base_c1_finetune")
print("claim_boundary=suggestive_exploratory_only")